# 🔬 Multi-Agent Debate & Consensus Fact-Checker

**Lightweight Google Colab launcher notebook.**

This notebook runs the entire modular repository end-to-end inside Google Colab without duplicating any `src/` code into notebook cells.

## Prerequisites
- Add your `GEMINI_API_KEY` to Colab Secrets (🔑 icon in left sidebar)
- Optionally add `GROQ_API_KEY` if using Groq as a fallback

In [ ]:
# Cell 1 — Environment Setup & Repository Access
import os

# Optional: Mount Google Drive to persist chroma_db/ and results/ across sessions
# from google.colab import drive
# drive.mount('/content/drive')

# Clone the repository (update URL to your repo)
REPO_URL = "https://github.com/YOUR_USERNAME/debate-fact-checker.git"
REPO_DIR = "/content/debate-fact-checker"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

In [ ]:
# Cell 2 — Dependency Installation
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — Secure API Key & Config Loading
import os

# Try Colab Secrets first, fallback to getpass
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ GEMINI_API_KEY loaded from Colab Secrets")
    
    # Optional: Groq fallback
    try:
        groq_key = userdata.get("GROQ_API_KEY")
        if groq_key:
            os.environ["GROQ_API_KEY"] = groq_key
            print("✅ GROQ_API_KEY loaded from Colab Secrets")
    except Exception:
        pass
        
except (ImportError, Exception):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY: ")
    print("✅ GEMINI_API_KEY loaded via getpass")

# Set LLM provider
os.environ["LLM_PROVIDER"] = "gemini"

# Verify (never print the actual key)
assert os.environ.get("GEMINI_API_KEY"), "❌ GEMINI_API_KEY not set!"
print(f"API key set: {'*' * 4}{os.environ['GEMINI_API_KEY'][-4:]}")

In [ ]:
# Cell 4 — Data Ingestion & Evaluation Pipeline
import os

# Step 1: Ingestion (skip if chroma_db/ already exists)
if not os.path.exists("chroma_db/chroma.sqlite3"):
    print("Running ingestion pipeline...")
    !python -m src.ingestion
else:
    print("✅ chroma_db/ already exists, skipping ingestion.")

# Step 2: Smoke test (10 claims) or full evaluation
SMOKE_TEST = True  # Set to False for full 60-claim evaluation

if SMOKE_TEST:
    !python -m src.evaluate --limit 10
else:
    !python -m src.evaluate

# Step 3: Display generated charts
from IPython.display import Image, display

charts_dir = "results/charts"
for chart_file in ["accuracy_f1_comparison.png", "per_class_f1.png", "confusion_matrices.png"]:
    chart_path = os.path.join(charts_dir, chart_file)
    if os.path.exists(chart_path):
        print(f"\n📊 {chart_file}")
        display(Image(filename=chart_path))
    else:
        print(f"⚠️ {chart_file} not found")

In [ ]:
# Cell 5 — Launch Live Gradio Demo
# Option A: CLI launch with public sharing
!python app.py --share

# Option B (alternative): Import and launch programmatically
# from app import create_app
# demo = create_app()
# demo.launch(share=True, debug=True)